<a href="https://colab.research.google.com/github/gjimenexv/Mineria-de-Datos/blob/master/pca_tsne_umap_store_sales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PCA, t-SNE y UMAP — Store Sales

Notebook sencillo para reducir dimensionalidad y visualizar **5.000 transacciones** de ventas con **gráficos interactivos (Plotly)**.

1. Cargar datos  
2. Convertir variables categóricas en **dummies**  
3. Estandarizar  
4. Aplicar **PCA**, **t-SNE** y **UMAP** (2D)  
5. Comparar los tres mapas  
6. **Casos de uso para el negocio** según los hallazgos

In [ ]:
# Solo si faltan librerías (Colab / entorno local)
%pip install -q pandas scikit-learn umap-learn plotly

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap


def plot_mapa_interactivo(coords, title, xlabel="Dim 1", ylabel="Dim 2"):
    """Scatter interactivo con Plotly: zoom, hover y leyenda clickeable."""
    tmp = pd.DataFrame({
        "x": coords[:, 0],
        "y": coords[:, 1],
        color_col: color_label,
        "Age": df["Age"],
        "Amount": df["Amount"],
        "ItemPurchased": df["ItemPurchased"],
    })
    fig = px.scatter(
        tmp, x="x", y="y", color=color_col,
        hover_data=[color_col, "Age", "Amount", "ItemPurchased"],
        title=title,
        labels={"x": xlabel, "y": ylabel},
        opacity=0.65,
        width=900, height=550,
    )
    fig.update_traces(marker=dict(size=6))
    fig.show()

## 1. Carga de datos

In [ ]:
df = pd.read_csv("store_sales.csv")
print(f"Filas: {len(df):,}  |  Columnas: {df.shape[1]}")
df.head()

Filas: 5,000  |  Columnas: 11


,CustomerID,Age,Gender,Category,ItemPurchased,Amount,Season,PaymentMethod,ItemRating,DiscountApplied(%),PreviousPurchases
0,1,58,Female,Accessories,Handbag,115.50,Autumn,Card,3.5,18,4
1,2,40,Male,Mens Clothing,Shirt,103.43,Spring,Card,4.1,13,4
2,3,66,Female,Sports,Football,35.45,Spring,Card,3.3,11,3
3,4,39,Female,Accessories,Handbag,153.31,Spring,Card,4.4,13,4
4,5,23,Female,Home,Curtains,151.43,Winter,Card,4.1,20,10


## 2. Preparación: dummies + estandarización

- Quitamos `CustomerID` (identificador, no aporta al análisis).
- Variables categóricas → **dummies** con `pd.get_dummies`.
- Estandarizamos con `StandardScaler` (recomendado para PCA y comparar escalas).

In [ ]:
# Columna para colorear los gráficos (no entra al modelo)
color_col = "Category"
color_label = df[color_col].copy()

# Quitar ID y separar numéricas / categóricas
df_model = df.drop(columns="CustomerID")
cat_cols = df_model.select_dtypes(include="object").columns
num_cols = df_model.select_dtypes(exclude="object").columns

# Dummies (drop_first evita columnas redundantes)
X = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

# Estandarizar
X_scaled = StandardScaler().fit_transform(X)

print(f"Variables originales: {len(num_cols)} numéricas + {len(cat_cols)} categóricas")
print(f"Matriz final para reducción: {X_scaled.shape}")

Variables originales: 5 numéricas + 5 categóricas
Matriz final para reducción: (5000, 47)


## 3. PCA

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords_pca = pca.fit_transform(X_scaled)

var = pca.explained_variance_ratio_ * 100
print(f"Varianza explicada — PC1: {var[0]:.1f}%  |  PC2: {var[1]:.1f}%  |  Total: {var.sum():.1f}%")

plot_mapa_interactivo(
    coords_pca, "PCA",
    xlabel=f"PC1 ({var[0]:.1f}%)", ylabel=f"PC2 ({var[1]:.1f}%)",
)

Varianza explicada — PC1: 7.0%  |  PC2: 5.3%  |  Total: 12.3%


### Interpretación del gráfico PCA

**Lo que se ve:** las 5.000 transacciones en 2 componentes que solo explican el **12,3%** de la varianza (PC1 ≈ 7,0%, PC2 ≈ 5,3%).

- **Electronics forma un grupo aislado hacia la derecha** (PC1 alto). Las 507 transacciones caen ahí sin mezclarse: el monto promedio es ~**1.766 USD** vs ~40–160 USD en el resto, y dominan móvil, laptop y smartwatch.
- **Beauty y Womens Clothing se superponen abajo a la izquierda** (~83 y ~113 USD de promedio, perfiles similares).
- **Footwear, Groceries y Home están muy mezclados en el centro-izquierda** (centroides casi encimados).
- **Sports y Mens Clothing suben un poco** (PC2 positivo), pero siguen solapándose con otras categorías.

**Conclusión:** en este dataset, **solo Electronics se separa claramente** por `Category`. El resto comparte un perfil de compra parecido en 2D.

## 4. t-SNE

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init="pca", learning_rate="auto")
coords_tsne = tsne.fit_transform(X_scaled)

plot_mapa_interactivo(coords_tsne, "t-SNE", xlabel="t-SNE 1", ylabel="t-SNE 2")

### Interpretación del gráfico t-SNE

**Lo que se ve:** el mismo dataset reorganizado para que puntos similares queden cerca (mapa no lineal).

- **Electronics sigue en una isla aparte**, hacia la derecha — la categoría más fácil de identificar.
- **En el centro hay una nube densa** donde se mezclan Footwear, Accessories, Home y Groceries (montos ~40–160 USD, mucha variabilidad interna).
- **Beauty queda más abajo**; **Sports y Mens Clothing tienden arriba**, pero los colores siguen entremezclados sin fronteras claras.
- La separación por `Category` **empeora respecto a PCA** (silueta ~0,22 vs ~0,33): más islas visuales, pero más mezcla en el centro.

**Conclusión:** t-SNE confirma que **Electronics es atípica**, pero **no descubre segmentos claros** para las otras 8 categorías.

## 5. UMAP

In [ ]:
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
coords_umap = reducer.fit_transform(X_scaled)

plot_mapa_interactivo(coords_umap, "UMAP", xlabel="UMAP 1", ylabel="UMAP 2")

### Interpretación del gráfico UMAP

**Lo que se ve:** proyección no lineal más continua, con `n_neighbors=15` y `min_dist=0.1`.

- **Electronics ya no domina** como en PCA/t-SNE: su nube se acerca al resto y comparte zona con Home, Beauty y Accessories.
- **Sports sube hacia la parte alta** (artículos deportivos), pero sigue mezclado con otros colores.
- **Mens Clothing y Womens Clothing se desplazan a la izquierda**, ligeramente separadas del bloque central.
- **El centro es un enjambre multicolor**: Groceries, Home, Footwear y Accessories comparten el mismo espacio.
- Es el mapa con **peor separación por categoría** (silueta ~0,21).

**Conclusión:** UMAP suaviza las diferencias. **Ninguna categoría forma un cluster limpio** salvo indicios débiles en Sports y ropa. `Category` no refleja grupos naturales fuertes en estos datos.

## 6. Comparación

In [ ]:
comparacion = pd.concat([
    pd.DataFrame({"x": c[:, 0], "y": c[:, 1], color_col: color_label, "Metodo": nombre})
    for nombre, c in [("PCA", coords_pca), ("t-SNE", coords_tsne), ("UMAP", coords_umap)]
])

fig = px.scatter(
    comparacion, x="x", y="y", color=color_col, facet_col="Metodo",
    hover_data=[color_col, "Metodo"],
    category_orders={"Metodo": ["PCA", "t-SNE", "UMAP"]},
    opacity=0.65, width=1100, height=450,
    title="Comparación PCA vs t-SNE vs UMAP",
)
fig.update_traces(marker=dict(size=4))
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

### Interpretación de la comparación

**Lo que se ve en los tres paneles** (misma leyenda, mismos 5.000 registros):

| Panel | Observación en *store_sales* |
|-------|------------------------------|
| **PCA** | **Electronics separada** a la derecha; resto mezclado. Es donde `Category` más se nota. |
| **t-SNE** | **Isla de Electronics** + nube central muy heterogénea. |
| **UMAP** | Mapa difuso; casi todas las categorías comparten zonas. |

**Conclusiones del ejemplo:**

1. **`Category` no define segmentos fuertes**: 8 de 9 categorías se solapan en los tres mapas.
2. **Electronics es la excepción** — se separa por monto muy alto (~1.766 USD vs ~40–160 USD) y productos tecnológicos. Es un outlier, no un patrón general.
3. **Ningún método descubre categorías ocultas**; las etiquetas actuales son demasiado amplias para agrupar perfiles distintos.
4. **PCA basta** para detectar la anomalía de Electronics. t-SNE y UMAP no mejoran la separación (silueta: PCA ~0,33 > t-SNE ~0,22 > UMAP ~0,21).

## 7. Casos de uso para el negocio

A partir de los mapas y las conclusiones de este análisis, estos son **escenarios concretos** donde el negocio puede actuar:

---

### 1. Segmento premium: Electronics

**Hallazgo:** Electronics es la única categoría que se separa de forma clara en todos los mapas (ticket promedio ~**1.766 USD** vs ~40–160 USD en el resto).

**Casos de uso:**
- **Programa VIP / retención:** identificar compradores de electrónica y ofrecerles garantía extendida, acceso anticipado a lanzamientos o financiamiento preferencial.
- **Cross-selling de alto valor:** recomendar accesorios compatibles (auriculares, smartwatch) en el checkout, donde el margen incremental es mayor.
- **Gestión de inventario:** tratar Electronics como categoría estratégica con stock dedicado y reglas de reposición distintas al resto del catálogo.
- **Detección de anomalías:** usar PCA como filtro rápido para flaggear transacciones atípicas (montos o combinaciones de producto fuera del perfil habitual de la categoría).

---

### 2. Categorías que se comportan igual: oportunidad de unificar estrategia

**Hallazgo:** Footwear, Groceries, Home, Accessories y parte de Sports/Mens Clothing se solapan en los tres métodos — comparten perfiles de compra similares (edad, descuento, temporada, monto medio).

**Casos de uso:**
- **Campañas de marketing conjuntas:** un mismo cupón de “compra general” puede funcionar para varias categorías sin perder relevancia (p. ej. descuento post-compra válido en Home + Groceries + Footwear).
- **Simplificar segmentación comercial:** en lugar de 9 segmentos por `Category`, el negocio puede operar con **2–3 macro-segmentos** (premium / estándar / básico) y reducir costos de personalización.
- **Bundles promocionales:** armar paquetes cruzados entre categorías que el mapa muestra juntas (ej. compra de cortinas + snacks + calzado en temporadas similares).

---

### 3. Beauty + Womens Clothing: cluster natural para marketing femenino

**Hallazgo:** ambas categorías se superponen en la zona inferior del mapa PCA y comparten rangos de monto (~83–113 USD).

**Casos de uso:**
- **Campaña unificada “belleza + moda”:** email, push o publicidad dirigida a mujeres con ofertas combinadas (perfume + top/falda).
- **Layout de tienda / e-commerce:** colocar Beauty cerca de Womens Clothing para favorecer compra impulsiva cruzada.
- **Programa de fidelización:** puntos dobles en compras que mezclen ambas categorías en la misma transacción o semana.

---

### 4. Limitaciones de `Category` como variable de segmentación

**Hallazgo:** 8 de 9 categorías no forman grupos naturales; t-SNE y UMAP no mejoran la separación respecto a PCA.

**Casos de uso (y qué hacer en su lugar):**
- **No usar solo `Category` para clustering de clientes** — el negocio necesitaría variables como frecuencia de compra, recencia, ticket acumulado o RFM.
- **Redefinir categorías de producto:** si el objetivo es segmentar, conviene agrupar por **ticket** (bajo / medio / alto) o por **tipo de necesidad** (tecnología, moda, hogar, ocio).
- **Priorizar PCA para dashboards ejecutivos:** es rápido, interpretable y suficiente para detectar el segmento atípico (Electronics) sin invertir en métodos más costosos.

---

### 5. Aplicaciones operativas del mapa interactivo

**Hallazgo:** los gráficos Plotly permiten explorar punto a punto (edad, monto, producto).

**Casos de uso:**
- **Equipos comerciales:** hacer zoom en una zona del mapa y revisar qué productos y montos concentran clientes similares antes de diseñar una promoción.
- **Product managers:** ocultar categorías en la leyenda para comparar visualmente dos segmentos (ej. Sports vs Mens Clothing) y decidir si merecen estrategias distintas.
- **Presentaciones a dirección:** usar el panel comparativo (PCA / t-SNE / UMAP) para explicar por qué una sola categoría requiere trato diferenciado y el resto puede gestionarse de forma agregada.

---

### Resumen ejecutivo

| Insight | Acción recomendada |
|---------|-------------------|
| Electronics aislada y de alto ticket | Tratarla como segmento premium con reglas propias |
| 8 categorías mezcladas | Unificar campañas o redefinir segmentos por comportamiento, no solo por categoría |
| Beauty + Womens Clothing juntas | Cross-selling y campañas femeninas integradas |
| PCA detecta lo esencial | Usar PCA en monitoreo rutinario; reservar t-SNE/UMAP para exploración puntual |

---

**Notas técnicas:** Plotly permite hover (Category, Age, Amount, ItemPurchased) y filtrar categorías desde la leyenda. Para colorear por otra variable: `color_col = "Gender"` y `color_label = df[color_col].copy()`.